## Processa CSVs iniciais
Ao final, salva arquivo para extração e classificação posterior

In [1]:
import pandas as pd
import re
from unidecode import unidecode

### Parte 0: carrega dados

In [2]:
# Carrega os dados enriquecidos
df = pd.read_csv('../data/silver/vagas_enriquecidas.csv', dtype=str).fillna('')
print('vagas enriquecidas', df.shape)

vagas enriquecidas (123737, 10)


In [3]:
# Carrega critérios
df_palavras_chave = pd.read_excel('data/criterios.xlsx', sheet_name='palavras_chave')
df_palavras_chave = df_palavras_chave.sort_values(by="Prioridade")

### Parte 1: prepara colunas

In [4]:
# Renomeia colunas para o padrão do pipeline
df_final = df.rename(columns={
    'Funcao': 'Cargo',
    'Localização': 'Local',
    'Média Salarial': 'Salario',
    'Skills Necessarias': 'Skills'
})
df_final = df_final[['Empresa', 'Cargo', 'Local', 'Salario', 'Skills']]

In [5]:
# Remove registros duplicados
print("Total", df_final.shape)
df_final = df_final.drop_duplicates()
print("Sem duplicadas", df_final.shape)

Total (123737, 5)


Sem duplicadas (123477, 5)


### Parte 2: Filtrando as vagas para cargos tech

In [6]:
# Processa a coluna "Cargo"
def prep_cargo(text):
    text = unidecode(text)  # Remove acentos
    text = text.lower()     # Converte para minúsculas
    text = text.replace('/', ' / ')
    text = text.replace('|', ' | ')
    text = text.replace('t.i', 'ti')
    text = text.replace('q.a', 'qa')
    text = text.replace('.', ' . ')
    text = text.replace('(', ' ( ')
    text = text.replace(')', ' ) ')
    text = text.replace('[', ' [ ')
    text = text.replace(']', ' ] ')
    text = text.replace(':', ' : ')
    text = text.replace('-', ' - ')
    text = text.replace(',', ' , ')
    text = re.sub(r'\s*\(\s*[aA]\s*\)\s*', ' ', text)
    text = re.sub(r'\s*\(\s*[oO]\s*\)\s*', ' ', text)
    return text

df_final["Cargo"] = df_final["Cargo"].apply(prep_cargo)

In [7]:
# Carrega as palavras chave
palavras_chave = df_palavras_chave['Cargo'].tolist()
palavras_chave = [" " + unidecode(palavra).lower() + " " for palavra in palavras_chave]

In [8]:
# Filtrando os cargos que contêm alguma das palavras-chave
regex_pattern = '|'.join(palavras_chave)
condition = df_final['Cargo'].str.contains(regex_pattern)
df_tech = df_final[condition].copy()
df_tech.shape

(15869, 5)

### Parte 3: Salva os dados

In [9]:
df_tech.to_csv('data/scp_01.csv', index=False)